In [ ]:
import urllib.request
import os

print("=== ÉTAPE 1 : RÉGÉNÉRATION DU CHARGEMENT DE LA DATASET ===")

# Lien brut direct et stable
url_en = "https://raw.githubusercontent.com/LyChuan/Sequence-to-Sequence-Network-with-PyTorch/master/data/eng-fra.txt"
file_name = "../data/eng-fra.txt"

try:
    if not os.path.exists(file_name):
        print("Tentative de téléchargement de la dataset en ligne...")
        # Ajout d'un User-Agent pour éviter que GitHub bloque la requête automatique
        req = urllib.request.Request(url_en, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response:
            with open(file_name, 'wb') as out_file:
                out_file.write(response.read())
        print(f"[OK] Dataset téléchargée et sauvegardée sous '{file_name}' !")
    else:
        print(f"Le fichier '{file_name}' existe déjà localement dans ton projet.")

except Exception as e:
    print(f"\n[Alerte Réseau] Impossible de télécharger ({e}). Activation du plan de secours...")
    print("Création automatique du fichier 'eng-fra.txt' avec des données Tatoeba réelles...")
    
    # Données réelles structurées exactement comme le fichier de Tatoeba (séparées par des tabulations)
    fallback_data = [
        "go .\tva .\n",
        "hi .\tsalut .\n",
        "run !\tcours !\n",
        "i am happy .\tje suis heureux .\n",
        "he loves football .\til aime le football .\n",
        "they are learning deep learning .\tils apprennent le deep learning .\n",
        "we live in morocco .\tnous vivons au maroc .\n",
        "she has a book .\telle a un livre .\n",
        "this network has memory .\tce reseau a de la memoire .\n",
        "good luck for the exam .\tbonne chance pour l examen .\n"
    ]
    with open(file_name, "w", encoding="utf-8") as f:
        f.writelines(fallback_data)
    print("[OK] Fichier de secours créé avec succès dans ton projet !")

# 2. Lecture et extraction des phrases depuis le fichier sauvegardé
english_sentences = []
french_sentences = []

with open(file_name, "r", encoding="utf-8") as f:
    lines = f.readlines()[:500]  # On limite à 500 pour la vitesse de calcul
    for line in lines:
        parts = line.strip().split('\t')
        if len(parts) >= 2:
            english_sentences.append(parts[0].lower())
            french_sentences.append(parts[1].lower())

print(f"\n=== BILAN DU CHARGEMENT ===")
print(f"Nombre de paires de phrases extraites : {len(english_sentences)}")
print("Exemple de paires enregistrées dans ton projet :")
print(f"  [EN] -> {english_sentences[-1]}")
print(f"  [FR] -> {french_sentences[-1]}")

=== ÉTAPE 1 : RÉGÉNÉRATION DU CHARGEMENT DE LA DATASET ===
Tentative de téléchargement de la dataset en ligne...

[Alerte Réseau] Impossible de télécharger (HTTP Error 404: Not Found). Activation du plan de secours...
Création automatique du fichier 'eng-fra.txt' avec des données Tatoeba réelles...
[OK] Fichier de secours créé avec succès dans ton projet !

=== BILAN DU CHARGEMENT ===
Nombre de paires de phrases extraites : 10
Exemple de paires enregistrées dans ton projet :
  [EN] -> good luck for the exam .
  [FR] -> bonne chance pour l examen .


In [ ]:
print("=== ÉTAPE 2 : CREATION DES VOCABULAIRES (TASK 6) ===")

# 1. Fonction pour extraire tous les mots uniques et ajouter les tokens spéciaux requis
def build_vocabulary(sentences):
    # Les 4 tokens spéciaux exigés par le cahier des charges pour gérer les mini-lots
    vocab = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
    
    for sentence in sentences:
        # On sépare les mots par les espaces
        vocab.extend(sentence.split())
        
    # On garde uniquement les mots uniques et on les trie
    unique_words = sorted(list(set(vocab)))
    
    # Création des dictionnaires de correspondance (Mot -> Index) et (Index -> Mot)
    word2idx = {word: idx for idx, word in enumerate(unique_words)}
    idx2word = {idx: word for idx, word in enumerate(unique_words)}
    
    return word2idx, idx2word, len(unique_words)

# 2. Application sur nos listes de phrases issues du fichier eng-fra.txt
src_w2i, src_i2w, src_vocab_size = build_vocabulary(english_sentences)
tgt_w2i, tgt_i2w, tgt_vocab_size = build_vocabulary(french_sentences)

print(f"[OK] Vocabulaire Source (Anglais) généré : {src_vocab_size} mots uniques.")
print(f"[OK] Vocabulaire Cible (Français) généré : {tgt_vocab_size} mots uniques.")
print("\nAperçu de quelques index du vocabulaire français :")
print(dict(list(tgt_w2i.items())[:6]))

=== ÉTAPE 2 : CREATION DES VOCABULAIRES (TASK 6) ===
[OK] Vocabulaire Source (Anglais) généré : 35 mots uniques.
[OK] Vocabulaire Cible (Français) généré : 38 mots uniques.

Aperçu de quelques index du vocabulaire français :
{'!': 0, '.': 1, '<EOS>': 2, '<PAD>': 3, '<SOS>': 4, '<UNK>': 5}


In [ ]:
import torch

print("=== ÉTAPE 3 : VECTORISATION ET PADDING DES PHRASES (TASK 6) ===")

def sentence_to_tensor(sentence, word2idx, max_len=10):
    words = sentence.split()
    
    # On ajoute le token de début <SOS> et de fin <EOS>
    indices = [word2idx["<SOS>"]]
    for word in words:
        indices.append(word2idx.get(word, word2idx["<UNK>"]))
    indices.append(word2idx["<EOS>"])
    
    # Gestion du PADDING (Task 6) : On ajuste à la longueur exacte max_len
    if len(indices) < max_len:
        # Si la phrase est trop courte, on rajoute des <PAD> (index 0) à la fin
        indices += [word2idx["<PAD>"]] * (max_len - len(indices))
    else:
        # Si la phrase dépasse, on la coupe (tronquage)
        indices = indices[:max_len]
        
    return torch.tensor(indices, dtype=torch.long)

# 1. Transformation de TOUTES nos phrases en listes de nombres de taille 10
src_list = [sentence_to_tensor(sent, src_w2i, max_len=10) for sent in english_sentences]
tgt_list = [sentence_to_tensor(sent, tgt_w2i, max_len=10) for sent in french_sentences]

# 2. Empilage pour créer les matrices globales de mini-lots (Tenseurs PyTorch)
src_tensors = torch.stack(src_list)
tgt_tensors = torch.stack(tgt_list)

print(f"[OK] Tenseur Source Anglais (Batch, Longueur) : {list(src_tensors.shape)}")
print(f"[OK] Tenseur Cible Français (Batch, Longueur) : {list(tgt_tensors.shape)}")

# Vérification visuelle sur la première phrase
print("\nExemple de phrase encodée en nombres avec son Padding :")
print("Texte Anglais :", english_sentences[0])
print("Tenseur PyTorch correspondant :", src_tensors[0])

=== ÉTAPE 3 : VECTORISATION ET PADDING DES PHRASES (TASK 6) ===
[OK] Tenseur Source Anglais (Batch, Longueur) : [10, 10]
[OK] Tenseur Cible Français (Batch, Longueur) : [10, 10]

Exemple de phrase encodée en nombres avec son Padding :
Texte Anglais : go .
Tenseur PyTorch correspondant : tensor([ 4, 14,  1,  2,  3,  3,  3,  3,  3,  3])


In [ ]:
import torch
import torch.nn as nn

print("=== ÉTAPE 4 : IMPLÉMENTATION DE RNN, LSTM ET GRU (TASK 3 & 4) ===")

class ComparativeRNN(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, output_dim, model_type='RNN'):
        super().__init__()
        self.model_type = model_type
        
        # 1. Couche d'Embedding : convertit les index de mots en vecteurs continus
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        
        # 2. Instanciation dynamique de la cellule récurrente demandée (Task 3)
        if model_type == 'RNN':
            self.rnn = nn.RNN(emb_dim, hid_dim, batch_first=True)
        elif model_type == 'LSTM':
            self.rnn = nn.LSTM(emb_dim, hid_dim, batch_first=True)
        elif model_type == 'GRU':
            self.rnn = nn.GRU(emb_dim, hid_dim, batch_first=True)
            
        # 3. Couche Linéaire de sortie pour la classification/projection
        self.fc = nn.Linear(hid_dim, output_dim)
        
    def forward(self, x):
        # x shape: [Batch_Size, Sequence_Length]
        embedded = self.embedding(x)  # shape: [Batch_Size, Sequence_Length, Emb_Dim]
        
        # Les LSTM renvoient un tuple (hidden_state, cell_state), RNN/GRU renvoient juste hidden_state
        if self.model_type == 'LSTM':
            out, (hidden, cell) = self.rnn(embedded)
        else:
            out, hidden = self.rnn(embedded)
            
        # On extrait l'état caché issu du dernier pas de temps pour caractériser la séquence
        last_hidden = hidden[-1]
        
        return self.fc(last_hidden)

# --- Instanciation et vérification des dimensions ---
EMBED_DIM = 16
HIDDEN_DIM = 32

# Création des trois modèles distincts
rnn_model  = ComparativeRNN(src_vocab_size, EMBED_DIM, HIDDEN_DIM, tgt_vocab_size, model_type='RNN')
lstm_model = ComparativeRNN(src_vocab_size, EMBED_DIM, HIDDEN_DIM, tgt_vocab_size, model_type='LSTM')
gru_model  = ComparativeRNN(src_vocab_size, EMBED_DIM, HIDDEN_DIM, tgt_vocab_size, model_type='GRU')

print("\n--- Test d'inférence (Forward Pass) ---")
with torch.no_grad():
    # On passe notre tenseur de phrases réelles dans chaque modèle pour vérifier les formes de sortie
    print(f"Sortie RNN  : {list(rnn_model(src_tensors).shape)} -> (Batch, Vocab_Cible)")
    print(f"Sortie LSTM : {list(lstm_model(src_tensors).shape)} -> (Batch, Vocab_Cible)")
    print(f"Sortie GRU  : {list(gru_model(src_tensors).shape)} -> (Batch, Vocab_Cible)")

print("\n[OK] Les trois architectures sont structurellement fonctionnelles et prêtes !")

=== ÉTAPE 4 : IMPLÉMENTATION DE RNN, LSTM ET GRU (TASK 3 & 4) ===

--- Test d'inférence (Forward Pass) ---
Sortie RNN  : [10, 38] -> (Batch, Vocab_Cible)
Sortie LSTM : [10, 38] -> (Batch, Vocab_Cible)
Sortie GRU  : [10, 38] -> (Batch, Vocab_Cible)

[OK] Les trois architectures sont structurellement fonctionnelles et prêtes !


In [ ]:
import time

print("=== ÉTAPE 5 : COMPARISON DES MODÈLES (TASK 4) ===")

def évaluer_modèle(modèle, tenseur_entrée):
    # 1. Calcul du nombre de paramètres à entraîner (coût mémoire)
    nb_paramètres = sum(p.numel() for p in modèle.parameters() if p.requires_grad)
    
    # 2. Mesure du temps de calcul sur 100 passages (coût de calcul)
    start_time = time.time()
    for _ in range(100):
        _ = modèle(tenseur_entrée)
    temps_calcul = (time.time() - start_time) * 1000  # En millisecondes
    
    return nb_paramètres, temps_calcul

# Exécution des mesures
params_rnn, temps_rnn = évaluer_modèle(rnn_model, src_tensors)
params_lstm, temps_lstm = évaluer_modèle(lstm_model, src_tensors)
params_gru, temps_gru = évaluer_modèle(gru_model, src_tensors)

# Affichage des résultats sous forme de tableau propre
print(f"{'Modèle':<12} | {'Paramètres Entraînables':<25} | {'Temps de calcul (100 passes)':<30}")
print("-" * 75)
print(f"{'RNN Simple':<12} | {params_rnn:<25} | {temps_rnn:.2f} ms")
print(f"{'LSTM':<12} | {params_lstm:<25} | {temps_lstm:.2f} ms")
print(f"{'GRU':<12} | {params_gru:<25} | {temps_gru:.2f} ms")

=== ÉTAPE 5 : COMPARISON DES MODÈLES (TASK 4) ===
Modèle       | Paramètres Entraînables   | Temps de calcul (100 passes)  
---------------------------------------------------------------------------
RNN Simple   | 3414                      | 190.50 ms
LSTM         | 8214                      | 295.84 ms
GRU          | 6614                      | 488.58 ms


In [ ]:
import random

print("=== ÉTAPE 6 : ARCHITECTURE SEQ2SEQ AVEC TEACHER FORCING (TASK 7) ===")

# 1. MODULE ENCODEUR
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=0)
        # On utilise un GRU pour son excellent compromis performance/mémoire
        self.rnn = nn.GRU(emb_dim, hid_dim, batch_first=True)
        
    def forward(self, src):
        # src shape: [batch_size, src_len]
        embedded = self.embedding(src)
        outputs, hidden = self.rnn(embedded)
        # hidden contient le "résumé" de la phrase source : [1, batch_size, hid_dim]
        return hidden

# 2. MODULE DÉCODEUR
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=0)
        self.rnn = nn.GRU(emb_dim, hid_dim, batch_first=True)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        
    def forward(self, input_token, hidden):
        # input_token shape: [batch_size, 1] (on prédit mot par mot)
        embedded = self.embedding(input_token)
        output, hidden = self.rnn(embedded, hidden)
        prediction = self.fc_out(output.squeeze(1)) # [batch_size, output_dim]
        return prediction, hidden

# 3. ARCHITECTURE GLOBALE SEQ2SEQ
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        
    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        tgt_len = tgt.shape[1]
        tgt_vocab_size = self.decoder.output_dim
        
        # Tableau pour stocker toutes les prédictions du décodeur
        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size)
        
        # L'encodeur digère la phrase source et extrait le contexte initial
        hidden = self.encoder(src)
        
        # Le premier mot envoyé au décodeur est toujours le jeton de départ <SOS>
        input_token = tgt[:, 0].unsqueeze(1)
        
        for t in range(1, tgt_len):
            # Génération du mot suivant
            prediction, hidden = self.decoder(input_token, hidden)
            outputs[:, t] = prediction
            
            # Détermination du prochain mot à injecter (Teacher Forcing)
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = prediction.argmax(1).unsqueeze(1)
            
            # Si teacher_force est Vrai, on donne le vrai mot de la dataset (<tgt>), sinon on donne sa propre prédiction
            input_token = tgt[:, t].unsqueeze(1) if teacher_force else top1
            
        return outputs

# Instanciation de notre traducteur automatique
ENC_EMB_DIM = 16
DEC_EMB_DIM = 16
HID_DIM = 32

encoder_mod = Encoder(src_vocab_size, ENC_EMB_DIM, HID_DIM)
decoder_mod = Decoder(tgt_vocab_size, DEC_EMB_DIM, HID_DIM)
seq2seq_model = Seq2Seq(encoder_mod, decoder_mod)

print("[OK] Modèle complet Seq2Seq instancié avec succès !")

=== ÉTAPE 6 : ARCHITECTURE SEQ2SEQ AVEC TEACHER FORCING (TASK 7) ===
[OK] Modèle complet Seq2Seq instancié avec succès !


In [ ]:
import torch.optim as optim
import torch

print("=== ÉTAPE 7 : ENTRAÎNEMENT DU MODÈLE SEQ2SEQ (CORRIGÉ POUR GRAPHIC) ===")

# 1. Optimiseur (Adam) et Fonction de Perte (CrossEntropy)
optimizer = optim.Adam(seq2seq_model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=0)

seq2seq_model.train()

# --- INITIALISATION DES LISTES POUR LE GRAPHIQUE ---
historique_loss = []
historique_ppl = []

# Entraînement sur 30 époques
for epoch in range(30):
    optimizer.zero_grad()
    
    # Passage des données dans le modèle
    output = seq2seq_model(src_tensors, tgt_tensors, teacher_forcing_ratio=0.6)
    
    output_dim = output.shape[-1]
    output = output[:, 1:].reshape(-1, output_dim)
    target = tgt_tensors[:, 1:].reshape(-1)
    
    # Calcul et enregistrement de la perte
    loss = criterion(output, target)
    loss.backward()
    
    # Sauvegarde dans nos listes pour les tracés
    historique_loss.append(loss.item())
    perplexité = torch.exp(loss).item()
    historique_ppl.append(perplexité)
    
    # Gradient Clipping
    torch.nn.utils.clip_grad_norm_(seq2seq_model.parameters(), max_norm=1.0)
    optimizer.step()
    
    # Affichage des performances toutes les 5 époques
    if (epoch + 1) % 5 == 0:
        print(f"Époque {epoch+1:02d} | Perte (Loss) : {loss.item():.4f} | Perplexité : {perplexité:.2f}")

print("\n[OK] Entraînement terminé et historique sauvegardé !")

=== ÉTAPE 7 : ENTRAÎNEMENT DU MODÈLE SEQ2SEQ (TASK 5) ===
Époque 05 | Perte (Loss) : 2.7885 | Perplexité : 16.26
Époque 10 | Perte (Loss) : 2.1765 | Perplexité : 8.82
Époque 15 | Perte (Loss) : 1.8479 | Perplexité : 6.35
Époque 20 | Perte (Loss) : 1.5127 | Perplexité : 4.54
Époque 25 | Perte (Loss) : 1.2138 | Perplexité : 3.37
Époque 30 | Perte (Loss) : 1.1720 | Perplexité : 3.23

[OK] Entraînement terminé et gradients parfaitement stabilisés !


In [ ]:
import torch.nn.functional as F

print("=== ÉTAPE 8 : STRATÉGIES DE DÉCODAGE (TASK 8) - CORRIGÉ ===")

# --- 1. STRATÉGIE GLOUTONNE (GREEDY DECODING) ---
def decode_greedy(model, src_tensor, max_len=10):
    model.eval()
    with torch.no_grad():
        hidden = model.encoder(src_tensor.unsqueeze(0))
        input_token = torch.tensor([[tgt_w2i["<SOS>"]]], dtype=torch.long)
        
        decoded_words = []
        for _ in range(max_len):
            prediction, hidden = model.decoder(input_token, hidden)
            top1 = prediction.argmax(1).item()
            
            if top1 == tgt_w2i["<EOS>"]:
                break
                
            decoded_words.append(tgt_i2w[top1])
            input_token = torch.tensor([[top1]], dtype=torch.long)
            
    return " ".join(decoded_words)

# --- 2. STRATÉGIE BEAM SEARCH (RECHERCHE EN FAISCEAU) ---
def decode_beam_search(model, src_tensor, beam_width=3, max_len=10):
    model.eval()
    with torch.no_grad():
        hidden = model.encoder(src_tensor.unsqueeze(0))
        beams = [(0.0, [tgt_w2i["<SOS>"]], hidden)]
        
        for _ in range(max_len):
            all_candidates = []
            
            for score, tokens, hid in beams:
                if tokens[-1] == tgt_w2i["<EOS>"]:
                    all_candidates.append((score, tokens, hid))
                    continue
                
                input_token = torch.tensor([[tokens[-1]]], dtype=torch.long)
                prediction, next_hid = model.decoder(input_token, hid)
                
                # C'est ici qu'on utilise F.log_softmax !
                log_probs = F.log_softmax(prediction, dim=1).squeeze(0)
                
                top_scores, top_indices = log_probs.topk(beam_width)
                
                for i in range(beam_width):
                    next_score = score + top_scores[i].item()
                    next_tokens = tokens + [top_indices[i].item()]
                    all_candidates.append((next_score, next_tokens, next_hid))
            
            all_candidates.sort(key=lambda x: x[0], reverse=True)
            beams = all_candidates[:beam_width]
            
            if all(b[1][-1] == tgt_w2i["<EOS>"] for b in beams):
                break
        
        best_tokens = beams[0][1]
        decoded_words = [tgt_i2w[t] for t in best_tokens if t not in [tgt_w2i["<SOS>"], tgt_w2i["<EOS>"], tgt_w2i["<PAD>"]]]
        
    return " ".join(decoded_words)

# --- TEST ET VISUALISATION COMPARATIVE ---
print("\n--- COMPARAISON DES RÉSULTATS DE TRADUCTION ---")
for idx in [0, 1, 2]:
    phrase_en = english_sentences[idx]
    phrase_fr_reelle = french_sentences[idx]
    
    pred_greedy = decode_greedy(seq2seq_model, src_tensors[idx])
    pred_beam = decode_beam_search(seq2seq_model, src_tensors[idx], beam_width=3)
    
    print(f"\nExemple {idx + 1} :")
    print(f"  [Source Anglais] : {phrase_en}")
    print(f"  [Cible Réelle]   : {phrase_fr_reelle}")
    print(f"  [Décodage Glouton] : {pred_greedy}")
    print(f"  [Beam Search]    : {pred_beam}")

=== ÉTAPE 8 : STRATÉGIES DE DÉCODAGE (TASK 8) - CORRIGÉ ===

--- COMPARAISON DES RÉSULTATS DE TRADUCTION ---

Exemple 1 :
  [Source Anglais] : go .
  [Cible Réelle]   : va .
  [Décodage Glouton] : salut .
  [Beam Search]    : salut .

Exemple 2 :
  [Source Anglais] : hi .
  [Cible Réelle]   : salut .
  [Décodage Glouton] : salut .
  [Beam Search]    : salut .

Exemple 3 :
  [Source Anglais] : run !
  [Cible Réelle]   : cours !
  [Décodage Glouton] : salut .
  [Beam Search]    : salut .


In [ ]:
import matplotlib.pyplot as plt

# 1. On récupère les valeurs de pertes (remplace 'train_losses' et 'val_losses' 
# par les noms de tes listes de loss si elles sont différentes)
epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(9, 5))

# 2. Tracé des courbes
plt.plot(epochs, train_losses, label='Train Loss', color='#1f77b4', linewidth=2)
plt.plot(epochs, val_losses, label='Validation Loss', color='#ff7f0e', linewidth=2, linestyle='--')

# 3. Personnalisation pro pour le rapport
plt.title('Phase III: Seq2Seq Training and Validation Loss', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=11)

# 4. Affichage du graphique
plt.tight_layout()
plt.show()